This notebok was created to combine `Hiking_journal.xlsx` and `Hikes.xlsx` in one dataframe and upload it to Notion.

In [ ]:
import pandas as pd
import os
import utils.utilities as uu
import numpy as np

path_to_source = uu.get_root_path()
path_to_source

PosixPath('/Users/olehbakumenko/Desktop/oleg/Workspaces/PytonWorkspace/harz_tsp/source')

In [ ]:
# Download the correct list from the google drive called Hiking_journal.xls
hiking_journal_df = pd.read_excel(
    os.path.join(path_to_source, "my_trips", "Hiking_journal.xlsx")
)
hiking_journal_df["Link"].ffill(inplace=True)
hiking_journal_df["tour_id"] = hiking_journal_df["Link"].apply(
    lambda x: x.split("/")[-1] if pd.notnull(x) else None
)
hikes_df = pd.read_excel(os.path.join(path_to_source, "my_trips", "Hikes.xlsx"))
hikes_df["tour_id"] = hikes_df["Link"].apply(
    lambda x: x.split("/")[-1] if pd.notnull(x) else None
)

todo_hikes_df = pd.read_csv(
    os.path.join(path_to_source, "my_trips", "Todo routes_all.csv")
)

export_hiking_joirnal_df = pd.DataFrame(columns=todo_hikes_df.columns)

export_hiking_joirnal_df = export_hiking_joirnal_df.rename(
    columns={
        "Name": "name",
        "Link": "link",
        "date_added": "date_visited",
        "estimated distance": "estimated_distance",
        "Nr. stamps": "nr_stamps",
    }
)


/var/folders/87/_jvy15c55mx0cfd6dqw9v64r0000gn/T/ipykernel_8538/623598056.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hiking_journal_df["Link"].ffill(inplace=True)


In [43]:
list(export_hiking_joirnal_df.columns)


['name',
 'link',
 'nr_stamps',
 'comment',
 'date_visited',
 'estimated_distance',
 'num_stamps',
 'transport_mode']

In [ ]:
update_dict_list = []

for _, group in hiking_journal_df.groupby("tour_id"):
    update_dict = {}
    tour_id = group["Link"].values[0].split("/")[-1]
    nr_stamps = np.sort(group["Nr. stamp"].tolist())
    date_visited = group["Date visited"].dt.strftime("%d.%m.%Y").values[0]
    date_visited = pd.to_datetime(date_visited, format="%d.%m.%Y")
    visited_by = group["Visited by"].dropna().values
    if len(visited_by) > 0:
        visited_by = visited_by.item()
    else:
        visited_by = None

    hike_row = hikes_df[hikes_df["tour_id"] == tour_id]
    update_dict.update(
        {
            "name": hike_row["Name"].values[0],
            "link": group["Link"].values[0],
            "nr_stamps": ", ".join([str(int(x)) for x in nr_stamps]),
            "date_visited": date_visited,
            "num_stamps": len(nr_stamps),
            "tour_id": tour_id,
            "estimated_distance": hike_row["Km"].values[0],
        }
    )
    if visited_by is not None:
        update_dict["visited_by"] = visited_by
    update_dict_list.append(update_dict)

In [55]:
export_hiking_joirnal_df = pd.DataFrame(update_dict_list)
export_hiking_joirnal_df

,name,link,nr_stamps,date_visited,num_stamps,tour_id,estimated_distance,visited_by
0,Sösetalsperre Rundgang (HWN 143) (Done),https://www.komoot.com/de-de/tour/1457237429,143,2024-03-03,1,1457237429,11.20,car
1,"Bad Harzburg Rundgang (HWN 169, 170, 122, 121,...",https://www.komoot.com/de-de/tour/1464742645,"121, 122, 169, 170",2024-03-10,4,1464742645,10.70,car
2,Barbis - Bismarkturm - Bad Lauterberg (HWN 151...,https://www.komoot.com/de-de/tour/1523646956,"101, 115, 151, 152",2024-04-20,4,1523646956,14.70,train
3,"Torfhaus Rundgang (HWN 135, 134, 149, 133, 221...",https://www.komoot.com/de-de/tour/1536927057,"133, 134, 135, 149, 221",2024-06-09,5,1536927057,16.50,train
4,"Okertal Rundgang (HWN 116, 118, 117) (Done)",https://www.komoot.com/de-de/tour/1538549011,"116, 117, 118",2024-04-28,3,1538549011,6.73,car
5,"Oderbrück - Radauwasserfall (HWN 168, 136, 019...",https://www.komoot.com/de-de/tour/1604902105,"1, 19, 136, 168",2024-05-26,4,1604902105,15.20,train
6,Torfhaus - Brocken - Oderbrück (HWN 009) (Done),https://www.komoot.com/de-de/tour/1604923318,9,2024-02-18,1,1604923318,18.20,train
7,"Goslar - Langelsheim (HWN 111,110, 109) (Done)",https://www.komoot.com/de-de/tour/1640184037,"109, 110, 111",2024-04-13,3,1640184037,13.50,train
8,"Clausthal-Zellerfeld (HWN 127, 145, 138, 137, ...",https://www.komoot.com/de-de/tour/1650062298,"127, 128, 137, 138, 145",2024-06-16,5,1650062298,15.60,car
9,"Wieda Rundgang (HWN 160, 58, 162 + Sonderstemp...",https://www.komoot.com/de-de/tour/1690779323,"58, 160, 162",2024-07-07,3,1690779323,11.40,car


In [56]:
export_hiking_joirnal_df.to_csv(
    os.path.join(path_to_source, "my_trips", "Hiking_journal_parsed.csv"), index=False
)